## **Implementación y refinamiento del modelo MLP — Detección de Fraude**

- Juan Pablo Solis
- Andre Yatmian Jo Mai

### 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

Dispositivo: cpu


### 2. Carga de datos y separación X / y

In [2]:
df = pd.read_csv('credit_card_fraud_selected_features.csv')

X = df.drop(columns=['Fraud_Flag']).values.astype(np.float32)
y = df['Fraud_Flag'].values.astype(np.float32)

print('Shape X:', X.shape)
print('Shape y:', y.shape)
print(f'Tasa de fraude: {y.mean()*100:.2f}%  →  dataset MUY desbalanceado')

Shape X: (500000, 10)
Shape y: (500000,)
Tasa de fraude: 1.50%  →  dataset MUY desbalanceado


### 3. Split train / validation / test

Se usa estratificación para mantener la proporción de fraude en cada partición.

In [3]:
# 70% train | 15% val | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print('Train:', X_train.shape, '| fraude:', y_train.mean().round(4))
print('Val:  ', X_val.shape,   '| fraude:', y_val.mean().round(4))
print('Test: ', X_test.shape,  '| fraude:', y_test.mean().round(4))

Train: (350000, 10) | fraude: 0.015
Val:   (75000, 10) | fraude: 0.015
Test:  (75000, 10) | fraude: 0.015


### 4. Manejo del desbalance de clases

Con solo 1.5% de fraude, un modelo naive que prediga siempre "no fraude" 
alcanzaría 98.5% de accuracy. Para evitar esto se usa **class_weight** que 
penaliza más los errores en la clase minoritaria (fraude).

In [4]:
classes = np.array([0, 1])
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print(f'Peso clase 0 (no fraude): {weights[0]:.4f}')
print(f'Peso clase 1 (fraude):    {weights[1]:.4f}')
print(f'→ El modelo penaliza {weights[1]/weights[0]:.0f}x más los errores en fraude')

Peso clase 0 (no fraude): 0.5076
Peso clase 1 (fraude):    33.3333
→ El modelo penaliza 66x más los errores en fraude


### 5. DataLoaders de PyTorch

In [5]:
BATCH_SIZE = 512

def make_loader(X, y, shuffle=True):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

Batches — Train: 684 | Val: 147 | Test: 147


### 6. Definición del modelo MLP

Arquitectura: **10 → 64 → 32 → 16 → 1**

- Activación ReLU en capas ocultas
- Dropout para regularización y evitar overfitting
- Salida con Sigmoid (probabilidad de fraude)

In [6]:
class FraudMLP(nn.Module):
    def __init__(self, input_dim=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model = FraudMLP(input_dim=X_train.shape[1]).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nParámetros totales: {total_params:,}')

FraudMLP(
  (net): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=16, bias=True)
    (7): ReLU()
    (8): Linear(in_features=16, out_features=1, bias=True)
    (9): Sigmoid()
  )
)

Parámetros totales: 3,329


### 7. Función de pérdida y optimizador

Se usa **BCELoss con pos_weight** para penalizar más los falsos negativos 
(fraudes no detectados), que son los más costosos en el mundo real.

In [7]:
# pos_weight = peso de la clase positiva (fraude)
criterion = nn.BCELoss(weight=None)  # usamos class_weights en el loop
pos_weight = torch.tensor([class_weights[1] / class_weights[0]]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Redefinir modelo sin Sigmoid final (BCEWithLogitsLoss incluye sigmoid internamente)
class FraudMLP(nn.Module):
    def __init__(self, input_dim=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model     = FraudMLP(input_dim=X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

print('Criterio:', criterion)
print('Optimizador:', optimizer)

Criterio: BCEWithLogitsLoss()
Optimizador: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 1e-05
)


### 8. Loop de entrenamiento con Early Stopping

In [8]:
EPOCHS        = 50
PATIENCE      = 7       # Early stopping: detener si val_loss no mejora
best_val_loss = np.inf
patience_cnt  = 0

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # --- ENTRENAMIENTO ---
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)

    # --- VALIDACIÓN ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            val_loss += criterion(model(X_batch), y_batch).item()
    val_loss /= len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'\nEarly stopping en epoch {epoch}')
            break

print(f'\nMejor val_loss: {best_val_loss:.4f}')

Epoch 01/50 | Train Loss: 1.3657 | Val Loss: 1.3651
Epoch 05/50 | Train Loss: 1.3645 | Val Loss: 1.3658
Epoch 10/50 | Train Loss: 1.3635 | Val Loss: 1.3673

Early stopping en epoch 10

Mejor val_loss: 1.3647


### 9. Curva de pérdida (Train vs Validation)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Época')
plt.ylabel('BCE Loss')
plt.title('Curva de pérdida — MLP Fraude')
plt.legend()
plt.tight_layout()
plt.show()

### 10. Evaluación en el conjunto de test

Se carga el mejor modelo guardado por Early Stopping.

In [ ]:
model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.eval()

all_probs, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# Umbral 0.5 por defecto
preds = (all_probs >= 0.5).astype(int)

print('=== Reporte de clasificación (umbral=0.5) ===')
print(classification_report(all_labels, preds, target_names=['No Fraude', 'Fraude']))
print(f'ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}')

### 11. Matriz de confusión

In [ ]:
cm = confusion_matrix(all_labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Fraude', 'Fraude'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión — Test Set')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Verdaderos Negativos  (TN): {tn:,}')
print(f'Falsos Positivos      (FP): {fp:,}')
print(f'Falsos Negativos      (FN): {fn:,}  ← fraudes no detectados')
print(f'Verdaderos Positivos  (TP): {tp:,}  ← fraudes detectados')

### 12. Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc = roc_auc_score(all_labels, all_probs)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'MLP (AUC = {auc:.4f})')
plt.plot([0,1],[0,1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC — MLP Fraude')
plt.legend()
plt.tight_layout()
plt.show()

### 13. Refinamiento — Ajuste de umbral de decisión

En detección de fraude, **el recall de fraude es más importante que el accuracy**.
Bajando el umbral de 0.5 se detectan más fraudes a costa de más falsos positivos.

In [ ]:
thresholds = [0.5, 0.4, 0.3, 0.2]

print(f"{'Umbral':>8} | {'Precision':>10} | {'Recall':>8} | {'F1':>8} | {'FN (fraudes perdidos)':>22}")
print('-' * 68)

for t in thresholds:
    p = (all_probs >= t).astype(int)
    report = classification_report(all_labels, p, output_dict=True, zero_division=0)
    fraud  = report.get('1', {})
    cm_t   = confusion_matrix(all_labels, p)
    fn_t   = cm_t[1][0]
    print(f"{t:>8.1f} | {fraud.get('precision',0):>10.4f} | {fraud.get('recall',0):>8.4f} | {fraud.get('f1-score',0):>8.4f} | {fn_t:>22,}")

print('\n→ Elegir el umbral según el costo de negocio: detectar más fraude vs. molestar más clientes legítimos.')